<a id="unit-u.fc85e8e6d07c4d83b51bb3cbc3466e55"></a>

<div>
  <div>1-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Doris-ке қосылу және алғашқы тапсырыстарды сұрау</div>
</div>

Тексеруге болатын он тарихи тапсырыспен Doris-ті үйренуді бастаңыз, содан кейін 5-модульде толық бизнес кестелерін жаппай жүктеңіз. Шамамен 25 минут жоспарлаңыз.

**1-модуль: Doris-пен танысу және алғашқы тапсырыстарды талдау** · [Сабақ](course.md) · [Тест](quiz01_doris_fundamentals.ipynb) · [Курстың README файлы](../../README.md)

> **Алғаш рет келдіңіз бе?** Бірінші ұяшықты орындамас бұрын курстың README файлындағы [Жұмысты бастау](../../README.md#get-started) бөлімін оқыңыз. Онда Python ортасы мен Docker түсіндіріледі, ал [Зертханалық жұмыс тәуелділіктері](../../README.md#lab-dependencies) бөлімінде қай жұмысқа не қажет екені көрсетілген.

Бұл зертханалық жұмыс Microsoft Wide World Importers жүйесінің модельденген көтерме сауда деректерін қолданады. Doris-ке он тапсырыс енгізіп, тапсырыс мәліметтерін сұрайсыз және салыққа дейінгі тапсырыс сомаларын күн бойынша қорытындылайсыз.

Үлгі файлдар MIT лицензиясымен курс құрамына енгізілген; курстың README файлындағы [Деректер жиыны](../../README.md#dataset) бөлімін қараңыз.


<a id="unit-u.0dc80b1f3db54c88bd21bd73e73d3be2"></a>

<a id="initialize-the-lab-tools"></a>

### Зертханалық құралдарды инициализациялау

Python құралдарын және көрсету мәнерлерін жүктеу үшін келесі ұяшықты орындаңыз. «Lab tools loaded» шыққанда, Doris-ті іске қосып, оған қосылу үшін 1-бөлімге өтіңіз. Ядроны қайта іске қосқаннан кейін осы жерден қайта орындаңыз.

ModuleNotFoundError шықса, тәуелділіктерді курстың README файлындағы [Жұмысты бастау](../../README.md#get-started) бөлімінде сипатталғандай орнатып, ноутбук курстың `.venv` ортасындағы ядроны қолданатынын тексеріңіз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.runtime import expect, fixture
from dw_course.wwi import HISTORY_COLUMNS, history_rows, sample
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.ui import card, install_styles

install_styles()
card("Continue to Section 1 and run the startup code to start and connect to Doris.", "ok", "Lab tools loaded")


<a id="unit-u.f8dedf33942e4222a11f78a65d2c654d"></a>

<a id="1-start-the-single-container-lab-environment"></a>

## 1. Бір контейнерлі зертханалық ортаны іске қосу

Бұл курс бір FE мен бір BE-ні бір контейнерде іске қосу үшін ресми All-in-One бейнесін пайдаланады.
Бұл сынақ ортасында сақтау мен есептеуі біріктірілген архитектура қолданылады. Ол өндірістік орналастыруға емес, оқуға арналған; кейінгі зертханалық жұмыстар сол контейнер мен зертханалық дерекқорды пайдалана береді.

Бастамас бұрын Docker Desktop (macOS) не Docker Engine (Linux) іске қосып, Compose плагинін орнатыңыз.
4 CPU, 8 GB жады және 20 GB бос диск орнын бөліңіз; бейнені алғаш жүктеу және іске қосу бірнеше минут алуы мүмкін.

**Төмендегі іске қосу кодын орындау курстың сынақ ортасын жасайды не іске қосады және зертханалық дерекқорға қосылады.**
Әдепкі зертханалық дерекқор — dw_course_l1_demo. 4-қадам ондағы **orders_sample** кестесін жойып, қайта жасайды;
басқа кестелер бұл бөлімнің қалпына келтіру аясына кірмейді. Алдымен осы оқу үлгісінің кестесін қалпына келтіруге болатынын растаңыз.

Курс құралдары қосылым мекенжайын автоматты баптайды; FE_HOST не FE_PORT енгізудің немесе қосылу әдісін таңдаудың қажеті жоқ.
**Docker мен Doris Jupyter ядросы орналасқан машинада жұмыс істейді; бұл браузеріңіз ашылған Mac болмауы мүмкін.**


<a id="unit-u.fb33c043fe21415f8649707dbf042f85"></a>

<a id="start-and-connect"></a>

### Іске қосу және қосылу

Келесі ұяшық Compose конфигурациясын тексереді, контейнерді іске қосады, жұмысқа жарамдылық тексерулерін күтеді, содан кейін FE қосылымын және BE есептеуін осы ретпен тексереді.
Курстың бұрыннан бар контейнері өз атаулы томдарын пайдалануды жалғастырады; метадеректер де, кесте деректері де жойылмайды және басқа қызметтер тоқтатылмайды.

Қосылғаннан кейін құралдар зертханалық дерекқорды жасап, таңдайды, сеанстың уақыт белдеуін +08:00 етіп және осы сеанс үшін `group_commit = 'off_mode'` орнатады.
Group Commit Doris-ке көптеген шағын жазуды бір транзакцияға біріктіруге мүмкіндік береді; ол өшірулі болса, әр жазу жеке бекітіледі, сондықтан олардың әрқайсысын бақылай аласыз. 5-модуль осы екеуін салыстырады. Кейінгі зертханалық жұмыстар осы сынақ ортасына тек қосылады; әрқайсысы бөлек контейнер іске қоспайды.


In [ ]:
prepare_environment(start=True)
lab = connect_sandbox()
lab.sql("SELECT 1 AS connection_ok, DATABASE() AS current_database", title="Connection and current database");


<a id="unit-u.f1b05416f79d49bf9e4c6b8f29e66c16"></a>

**Күтілетін нәтиже:** connection_ok мәні 1, ал әдепкі current_database — dw_course_l1_demo.

**Ұяшық тоқтаса**, оның карточкасы себебін көрсетеді:

- Docker орнатылмаған немесе жұмыс істемейді: Docker Desktop не Docker Engine іске қосыңыз, терминалда `docker info` сәтті орындалғанша күтіңіз, содан кейін осы ұяшықты қайта орындаңыз.
- Порт бос емес: карточка портты және оны орнататын айнымалыны көрсетеді, мысалы `52030 (DW_FE_QUERY_PORT, FE MySQL protocol)`, әрі ештеңе іске қосылмайды. Бағдарламаны `lsof -nP -iTCP:52030 -sTCP:LISTEN` арқылы тауып, ол өзіңіздікі болса, тоқтатыңыз. Әйтпесе курсқа басқа порттар беріңіз: Jupyter-ді тоқтатып, айнымалыларды орнатып қайта іске қосыңыз, мысалы `DW_FE_QUERY_PORT=62030 DW_FE_HTTP_PORT=61030 DW_BE_HTTP_PORT=61040 .venv/bin/jupyter lab`, содан кейін осы зертханалық жұмысты басынан орындаңыз. Әр зертханалық жұмыс бірдей айнымалыларды оқиды, сондықтан әр жолы Jupyter-ді солармен іске қосыңыз.
- Жұмысқа жарамдылық тексеруінің күту уақыты аяқталды: курс каталогынан `docker compose --project-name doris-warehousing-course --file environments/single-node/compose.yml logs --tail 100 doris` арқылы контейнер журналын оқып, осы ұяшықты қайта орындаңыз. Қайта орындау контейнерді және оның деректерін қайта пайдаланады.

Курстың README файлындағы [Орта](../../README.md#environment) бөлімінде порттар, олардың айнымалылары және курстың Docker нысандары берілген.


<a id="unit-u.48478b79f7be45d385abcc3f4982e79f"></a>

<a id="2-check-fe-and-be"></a>

## 2. FE мен BE-ні тексеру

FE SQL қабылдап, сұрауларды жоспарлайды; BE оларды орындайды және осы сақтау мен есептеуі біріктірілген ортада ішкі кестелердің деректерін де сақтайды.
Деректерді жазбас бұрын түйіндер жұмыс істеп тұрғанын растаңыз.


In [ ]:
lab.sql("SELECT VERSION() AS protocol_version, @@version_comment AS version_comment", title="Connection version information");
lab.sql("SHOW FRONTENDS", title="FE nodes", columns=["Name", "Host", "Role", "IsMaster", "Alive", "Version"]);
lab.sql("SHOW BACKENDS", title="BE nodes", columns=["BackendId", "Host", "HttpPort", "Alive", "TabletNum", "Version"]);

<a id="unit-u.4163763660fc40c7aeb236c531c6f2d3"></a>

**Күтілетін нәтиже:** бір FE және бір BE, екеуінде де `Alive` мәні true; FE — master. Екеуі де `Version` өрісінде `doris-4.1.3-rc02-7126cf65d96` көрсетеді: Apache Doris 4.1.3 шығарылымы сол шығарылым кандидатының commit-іне сәйкес келеді. `SELECT VERSION()` Doris нұсқасын емес, Doris MySQL клиенттеріне хабарлайтын MySQL хаттамасының `5.7.99` нұсқасын қайтарады; `@@version_comment` те Doris нұсқасын атайды.

BE жұмыс істемесе, алдымен орта мәселесін шешіңіз. Істеп кетер ме екен деп кестелерді жасап, деректерді енгізе бермеңіз.


<a id="unit-u.1769fa4102644fac98185700e41dcb27"></a>

<a id="3-understand-the-wwi-order-sample"></a>

## 3. WWI тапсырыс үлгісін түсіну

WWI Orders ішінен 2013-01-01 және 2013-01-02 күндерінің әрқайсысына бес тапсырыс таңдалады, ал OrderLines тапсырыс бойынша агрегатталып, әр тапсырысқа бір жолы бар кіріспе үлгі жасалады. 5-модуль бастапқы көп кестелі деректерді жүктейді.

| Өріс | Мағынасы |
| --- | --- |
| order_id, customer_id | Бастапқы WWI бизнес идентификаторлары |
| order_date | Бастапқы тапсырыс күні |
| order_amount | Егжей-тегжейлі жолдар бойынша Quantity × UnitPrice қосындысы, салыққа дейінгі тапсырыс сомасын білдіреді |
| line_count | Тапсырыстың егжей-тегжейлі тауар жолдарының саны |
| data_source | Дереккөзі, осы пакет үшін WWI |

Он тапсырыстың салыққа дейінгі жалпы сомасы — 12220.60. Төлем күйі үшін бөлек төлем немесе шот деректері қажет.
Деректер файлы: [wwi/sample.json](../../datasets/wwi/sample.json); тапсырыс идентификаторлары — 1–5 және 80–84.


<a id="unit-u.642d617023fd428092fe66eca96c6d76"></a>

<a id="4-create-your-first-internal-table"></a>

## 4. Алғашқы ішкі кестені жасау

Толық DDL төменде көрсетілген: DECIMAL сомаларды, ал DATE бастапқы тапсырыс күндерін сақтайды.
DUPLICATE KEY(order_id) сұрыптау үшін қолданылады және бірегейлікті қамтамасыз етпейді; осы бөлімнің шағын үлгісі үшін бір бакет және бір реплика пайдаланылады.

**Қалпына келтіру ескертуі:** келесі ұяшық бүкіл дерекқорды емес, ағымдағы зертханалық дерекқордағы тек orders_sample кестесін жояды.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_sample")
lab.execute("""
CREATE TABLE orders_sample (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_date DATE NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL,
    line_count INT NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.sql("DESC orders_sample", title="Historical order projection fields");
lab.sql("SHOW CREATE TABLE orders_sample", title="Actual table definition");


<a id="unit-u.537b9ddfd8134dbb895b4fa15617c0e7"></a>

**Күтілетін нәтиже:** алты өріс, Duplicate Key, бір бакет және бір реплика. Кесте жасалды, бірақ оған әлі деректер енгізілген жоқ.


<a id="unit-u.579f95408b8a4012921aa12c397ad7b3"></a>

<a id="5-insert-ten-historical-orders"></a>

## 5. Он тарихи тапсырысты енгізу

Айқын INSERT өрістерді түсінуді жеңілдетеді; 5-модуль барлық бизнес кестелерін файлдардан жүктейді. Деректер WWI жүйесінен алынған, сомалар бастапқы егжей-тегжейлі жолдардан агрегатталған.
Бұл ұяшықты жеке қайта орындамаңыз: Duplicate Key деректерді қосады. Бүкіл процесті қайта орындау үшін алдымен 4-қадамды орындаңыз.


In [ ]:
lab.execute("""
INSERT INTO orders_sample (
    order_id, customer_id, order_date, order_amount, line_count, data_source
) VALUES
(1, 832, '2013-01-01', 2300.00, 1, 'WWI'),
(2, 803, '2013-01-01', 405.00, 2, 'WWI'),
(3, 105, '2013-01-01', 90.00, 1, 'WWI'),
(4, 57, '2013-01-01', 445.20, 3, 'WWI'),
(5, 905, '2013-01-01', 704.00, 3, 'WWI'),
(80, 543, '2013-01-02', 1138.00, 3, 'WWI'),
(81, 456, '2013-01-02', 376.00, 2, 'WWI'),
(82, 487, '2013-01-02', 234.00, 2, 'WWI'),
(83, 154, '2013-01-02', 6220.40, 4, 'WWI'),
(84, 111, '2013-01-02', 308.00, 2, 'WWI')
""")
lab.sql("SELECT * FROM orders_sample ORDER BY order_id", title="Check the ten WWI orders");


<a id="unit-u.7d35db289f9f45d185c674a7c025b1b9"></a>

**Күтілетін нәтиже:** 1–5 және 80–84 тапсырыс идентификаторлары, барлығы он жол; 1-тапсырыстың салыққа дейінгі сомасы — 2300.00. Жиырма жол шықса, INSERT қайталанғанын тексеріп, белгіленген аяда кестені қайта жасаңыз.


<a id="unit-u.ff7dac253eb14f02ade3d6e9e9c5810b"></a>

<a id="6-check-both-totals-and-details"></a>

## 6. Жалпы сомалар мен егжей-тегжейлі деректерді тексеру

COUNT(*) — проекциядағы жолдар саны; ол тек әр тапсырысқа бір жол келгенде ғана тапсырыстар санына тең. OrderLines жолдарын тікелей санау егжей-тегжейлі тауар жолдарының санын береді.
Жалпы сомалардың кездейсоқ сәйкес келуі қателерді жасырмауы үшін әр өрісті репозиторийдегі бекітілген үлгімен салыстырыңыз.


In [ ]:
lab.sql("SELECT COUNT(*) AS order_count, SUM(order_amount) AS order_amount FROM orders_sample", title="Pre-tax order amount");
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample"), [(10, "12220.60")],
       title="Ten orders, 12220.60")
expect(lab.query("""
SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source
FROM orders_sample ORDER BY order_id
"""), history_rows(), title="Every field matches the sample")

<a id="unit-u.d3a797586650447b822826f7aeb28a07"></a>

**Күтілетін нәтиже:** он жол, 12220.60, барлық өріс сәйкес келеді. Бұл нәтиже алынған төлемдерді, салық қосылған шот-фактура сомаларын немесе пайданы емес, тек салыққа дейінгі тапсырыс сомасын білдіреді.


<a id="unit-u.d8d59c03ae4845e8bcf9276dfd73ba62"></a>

<a id="7-answer-the-business-question-daily-orders"></a>

## 7. Бизнес сұрағына жауап беру: күндік тапсырыстар

Он үлгі тапсырыстың саны мен сомасы екі күнге қалай бөлінетінін көру үшін тапсырыс күні бойынша топтап, сұрыптаңыз. Қамту аясы осы бөлімнің он тапсырысымен шектеледі.


In [ ]:
daily_sql = """
SELECT CAST(order_date AS STRING) AS order_date, COUNT(*) AS order_count,
       SUM(order_amount) AS order_amount
FROM orders_sample GROUP BY order_date ORDER BY order_date
"""
lab.sql(daily_sql, title="Daily summary of sample orders");
expect(lab.query(daily_sql), [("2013-01-01", 5, "3944.20"), ("2013-01-02", 5, "8276.40")],
       title="Five orders a day: 3944.20 and 8276.40")

<a id="unit-u.c53a6b9c12fa4c279a7fcd10b29c6595"></a>

**Күтілетін нәтиже:** 2013-01-01: бес тапсырыс, 3944.20; 2013-01-02: бес тапсырыс, 8276.40.


<a id="unit-u.7c5e32f58b234161ac9c69a99f904863"></a>

<a id="8-practice-orders-of-at-least-100000"></a>

## 8. Жаттығу: сомасы кемінде 1000.00 болатын тапсырыстар

Салыққа дейінгі сомасы кемінде 1000.00 болатын тапсырыстардың `order_id`, `order_date` және `order_amount` өрістерін тапсырыс идентификаторы бойынша сұрыптап қайтаратын сұрау жазыңыз. Жалпы сомасы 9658.40 болатын үш тапсырыс күтіледі.


In [ ]:
# Your code here

<a id="unit-u.0fe001406e2a44279e40e2c25a6be9c1"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.sql("""
SELECT order_id, order_date, order_amount FROM orders_sample
WHERE order_amount >= 1000.00 ORDER BY order_id
""", title="Amount filter: reference solution");
expect(lab.query("SELECT order_id FROM orders_sample WHERE order_amount >= 1000.00 ORDER BY order_id"),
       [(1,), (80,), (83,)], title="Orders 1, 80, and 83")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample WHERE order_amount >= 1000.00"),
       [(3, "9658.40")], title="Three orders, 9658.40")

<a id="unit-u.b37c5c78c20b48a68cbdb2fede5cc152"></a>

**Күтілетін нәтиже:** 1 (2300.00), 80 (1138.00) және 83 (6220.40) тапсырыстары және екі жасыл белгі: үш тапсырыс, 9658.40. `>=` дәл 1000.00 сомасы бар тапсырысты сақтайды; `>` оны сақтамайды.


<a id="unit-u.10adc9a5383f40fba756b11bd967a221"></a>

<a id="9-practice-two-conditions"></a>

## 9. Жаттығу: екі шарт

2013-01-02 күніндегі сомасы кемінде 300 болатын тапсырыстарды санап, салыққа дейінгі сомаларын қосыңыз. Сұрауды жазбас бұрын WHERE ішінде қандай шарттар болуы керегін анықтаңыз. 4 тапсырыс және 8042.40 күтіледі.

Кодты келесі ұяшыққа жазыңыз, содан кейін одан кейінгі үлгі шешіммен салыстырыңыз.


In [ ]:
# Your code here

<a id="unit-u.8088cf1934de4160861ac8c8a77c9374"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
query = """SELECT COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_sample
WHERE order_date = '2013-01-02' AND order_amount >= 300"""
lab.sql(query, title="Orders on the second day with an amount of at least 300");
expect(lab.query(query), [(4, "8042.40")], title="Date and amount filters are correct")

<a id="unit-u.a9b6230cf8fd44618b39f1c7afa6d2ed"></a>

**Күтілетін нәтиже:** 4 тапсырыс және 8042.40, жасыл белгі. Екі шарт та `WHERE` ішінде `AND` арқылы біріктірілуі керек: күн мен сома жолдарды санаудан бұрын сүзеді. 234.00 сомасы бар 82-тапсырыс — екінші күнде сома шарты алып тастайтын тапсырыс.


<a id="unit-u.8152875676fb4c31a0cde7b3581acf07"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- Doris-ке қосылып, бір жұмыс істеп тұрған FE мен бір жұмыс істеп тұрған BE көрдіңіз және Doris нұсқасын қайдан қарау керегін білесіз;
- `orders_sample` кестесінде он тапсырыс және 12220.60 сомасы бар, әр өріс үлгіге сәйкес келеді;
- күндік қорытындының әр бөлігін: `GROUP BY`, `COUNT(*)`, `SUM` және `ORDER BY` түсіндіре аласыз.

Толық қайта орындау үшін инициализациядан бастаңыз; бұл бөлім тек orders_sample кестесін қайта жасайды. Ядроны өшіру қосылымды босатады, ал курс деректері зертханалық дерекқорда қалады.


<a id="unit-u.b80e7f4fbe6a461cb74c8e46fd90151f"></a>

<a id="next"></a>

## Келесі қадам

[1-тест: Doris-пен танысу және алғашқы тапсырыстарды талдау](quiz01_doris_fundamentals.ipynb) файлын ашыңыз, содан кейін [2-модуль: Doris сақтау архитектурасы және жазу механизмдері](../module02-architecture/course.md) бөліміне өтіңіз.
